# Create Dataset Splits For Training

This notebook turns `openui_statistics_samples.jsonl` into readable per-sample JSON files split 8/1/1 across `training/data/train`, `training/data/valid`, and `training/data/test`.

In [ ]:
from __future__ import annotations

import json
import random
import re
import shutil
from pathlib import Path
from typing import Any

SPLIT_RATIOS = {"train": 0.8, "valid": 0.1, "test": 0.1}
RANDOM_SEED = 42
SOURCE_NAME = "openui_statistics_samples.jsonl"


## Resolve Paths

In [ ]:
def find_generated_dir(start=None):
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        direct = candidate / SOURCE_NAME
        nested = candidate / "training" / "data" / "generated" / SOURCE_NAME
        if direct.exists():
            return candidate
        if nested.exists():
            return nested.parent
    raise FileNotFoundError(f"Could not find {SOURCE_NAME} from {start}")


GENERATED_DIR = find_generated_dir()
DATA_DIR = GENERATED_DIR.parent
SOURCE_PATH = GENERATED_DIR / SOURCE_NAME

print(f"Generated dir: {GENERATED_DIR}")
print(f"Source: {SOURCE_PATH}")
print(f"Split output root: {DATA_DIR}")


## Load Samples

In [ ]:
def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for line_no, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path}:{line_no}") from exc
    return rows


samples = read_jsonl(SOURCE_PATH)
print(f"Loaded {len(samples)} samples")
samples[0]


## Build 8/1/1 Splits

In [ ]:
def shuffled_indices(size, seed):
    indices = list(range(size))
    random.Random(seed).shuffle(indices)
    return indices


def split_indices(size):
    indices = shuffled_indices(size, RANDOM_SEED)
    train_end = int(size * SPLIT_RATIOS["train"])
    valid_end = train_end + int(size * SPLIT_RATIOS["valid"])
    return {
        "train": indices[:train_end],
        "valid": indices[train_end:valid_end],
        "test": indices[valid_end:],
    }


splits = split_indices(len(samples))
{name: len(indices) for name, indices in splits.items()}


## Write Pretty JSON Files

In [ ]:
def slugify(value, fallback="sample"):
    text = (value or fallback).lower()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")[:80] or fallback


def sample_filename(sample, source_index, split_position):
    query_result = sample.get("query_result") or {}
    package = slugify(query_result.get("package_name") or query_result.get("dataset_title"))
    resource = slugify(query_result.get("resource_name"), fallback="resource")
    return f"{split_position:04d}-{source_index:04d}-{package}-{resource}.json"


def reset_split_dirs(data_dir, split_names):
    for split_name in split_names:
        split_dir = data_dir / split_name
        if split_dir.exists():
            shutil.rmtree(split_dir)
        split_dir.mkdir(parents=True, exist_ok=True)


def write_split_files(samples, splits, data_dir):
    reset_split_dirs(data_dir, list(splits))
    manifest = {
        "source": str(SOURCE_PATH.relative_to(DATA_DIR.parent)),
        "seed": RANDOM_SEED,
        "ratios": SPLIT_RATIOS,
        "splits": {},
    }

    for split_name, indices in splits.items():
        split_dir = data_dir / split_name
        files = []
        for position, source_index in enumerate(indices, start=1):
            sample = samples[source_index]
            filename = sample_filename(sample, source_index, position)
            path = split_dir / filename
            path.write_text(json.dumps(sample, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            files.append({"file": filename, "source_index": source_index})

        split_manifest = {
            "split": split_name,
            "count": len(files),
            "files": files,
        }
        (split_dir / "manifest.json").write_text(
            json.dumps(split_manifest, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        manifest["splits"][split_name] = {"count": len(files), "dir": str(split_dir.relative_to(DATA_DIR.parent))}

    (data_dir / "openui_statistics_splits.manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return manifest


manifest = write_split_files(samples, splits, DATA_DIR)
manifest


## Quick Integrity Check

In [ ]:
written_counts = {
    split_name: len([path for path in (DATA_DIR / split_name).glob("*.json") if path.name != "manifest.json"])
    for split_name in splits
}
assert sum(written_counts.values()) == len(samples), written_counts
assert written_counts == {name: len(indices) for name, indices in splits.items()}
written_counts
